In [2]:
import time
from IPython import get_ipython

ip = get_ipython()

def pre_run_cell(info):
    ip.user_ns["_start_time"] = time.time()

def post_run_cell(result):
    start_time = ip.user_ns.get("_start_time", None)
    if start_time is not None:
        duration = time.time() - start_time
        print(f"\n Cell execution time: {duration:.4f} seconds")

ip.events.register("pre_run_cell", pre_run_cell)
ip.events.register("post_run_cell", post_run_cell)

print(" Cell execution timer is now active.")


 Cell execution timer is now active.


In [ ]:
! pip install qiskit qiskit-aer qiskit-machine-learning


In [3]:
from qiskit.providers.aer import AerSimulator
from qiskit_machine_learning.neural_networks import CircuitQNN
print("Imports successful!")


Imports successful!

 Cell execution time: 0.0114 seconds


iris dataset - QNN

In [4]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelBinarizer
from qiskit import Aer
from qiskit.providers.aer import AerSimulator
from qiskit.circuit import Parameter, QuantumCircuit
from qiskit import execute

# Step 1: Load Iris dataset
data = load_iris()
X = data.data
y = data.target

# Step 2: Preprocess the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# One-hot encode the target labels
lb = LabelBinarizer()
y_one_hot = lb.fit_transform(y)

# Step 3: Split the data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_one_hot, test_size=0.3, random_state=42)

# Step 4: Create quantum circuit
n_qubits = 3
params = [Parameter(f'θ{i}') for i in range(2 * n_qubits)]
qc = QuantumCircuit(n_qubits)
for i in range(n_qubits):
    qc.rx(params[2*i], i)
    qc.ry(params[2*i+1], i)
qc.measure_all()

# Step 5: Quantum simulator
simulator = AerSimulator()

# Step 6: Circuit execution
def run_circuit_with_params(params_values):
    param_dict = {params[i]: params_values[i] for i in range(len(params))}
    bound_circuit = qc.bind_parameters(param_dict)
    result = execute(bound_circuit, simulator, shots=1024).result()
    counts = result.get_counts()
    zeros = sum(count for bitstring, count in counts.items() if bitstring.endswith('0'))
    ones = sum(count for bitstring, count in counts.items() if bitstring.endswith('1'))
    prediction = 0 if zeros > ones else 1
    return prediction

# Step 7: Accuracy evaluation
def evaluate_accuracy(X_data, y_data):
    correct = 0
    for i in range(len(X_data)):
        params_values = np.tile(X_data[i], 2)[:len(params)]
        prediction = run_circuit_with_params(params_values)
        correct += (prediction == np.argmax(y_data[i]))
    return correct / len(X_data)

# Step 8: Compute and print accuracies
train_acc = evaluate_accuracy(X_train, y_train)
test_acc = evaluate_accuracy(X_test, y_test)

print(f"Training Accuracy: {train_acc:.2f}")
print(f"Test Accuracy: {test_acc:.2f}")


Training Accuracy: 0.25
Test Accuracy: 0.31

 Cell execution time: 3.0968 seconds


Rain dataset - QNN

In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from qiskit import Aer, execute
from qiskit.providers.aer import AerSimulator
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# Step 1: Load Rain dataset
rain_data = pd.read_excel(r'C:\Users\gobig\qiskit\rain_dataset\rain_dataset.xlsx')
rain_data = rain_data.head(100)

# Step 2: Select features and target
features = ['MinTemp', 'Humidity9am', 'WindSpeed3pm', 'Pressure9am', 'WindDir9am']
target = 'RainTomorrow'
X = rain_data[features].copy()
y = rain_data[target].map({'Yes': 1, 'No': 0}).values

# Step 3: Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['MinTemp', 'Humidity9am', 'WindSpeed3pm', 'Pressure9am']),
        ('cat', OneHotEncoder(), ['WindDir9am'])
    ])
X_scaled = preprocessor.fit_transform(X).toarray()

# Step 4: Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# Step 5: Quantum Circuit
n_qubits = X_train.shape[1]
params = [Parameter(f'θ{i}') for i in range(2 * n_qubits)]
qc = QuantumCircuit(n_qubits)
for i in range(n_qubits):
    qc.rx(params[2 * i], i)
    qc.ry(params[2 * i + 1], i)
qc.measure_all()

simulator = AerSimulator()

# Step 6: Run circuit
def run_circuit_with_params(params_values):
    param_dict = {params[i]: params_values[i] for i in range(len(params))}
    bound_circuit = qc.bind_parameters(param_dict)
    result = execute(bound_circuit, simulator, shots=1024).result()
    counts = result.get_counts()
    zeros = sum(c for bit, c in counts.items() if bit.endswith('0'))
    ones = sum(c for bit, c in counts.items() if bit.endswith('1'))
    prediction = 0 if zeros > ones else 1
    return prediction

# Step 7: Accuracy evaluation
def evaluate_accuracy(X_data, y_data):
    correct = 0
    for i in range(len(X_data)):
        params_values = np.tile(X_data[i], 2)[:len(params)]
        prediction = run_circuit_with_params(params_values)
        correct += (prediction == y_data[i])
    return correct / len(X_data)

# Step 8: Output results
train_acc = evaluate_accuracy(X_train, y_train)
test_acc = evaluate_accuracy(X_test, y_test)

print(f"Training Accuracy: {train_acc:.2f}")
print(f"Test Accuracy: {test_acc:.2f}")


Training Accuracy: 0.84
Test Accuracy: 0.83

 Cell execution time: 72.4540 seconds


Vlds dataset - QNN

In [6]:
import numpy as np
from sklearn.datasets import make_multilabel_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from qiskit import QuantumCircuit, Aer
from qiskit.utils import algorithm_globals
from qiskit_machine_learning.algorithms import VQC
from qiskit.algorithms.optimizers import COBYLA  #  Correct optimizer import
from qiskit.circuit import ParameterVector


# Set random seed for reproducibility
algorithm_globals.random_seed = 42

# Step 1: Generate Vlds dataset
X, y = make_multilabel_classification(n_samples=100, n_features=5, n_classes=1, 
                                     n_labels=1, random_state=42)
y = y.ravel()  # Flatten to 1D array

# Step 2: Preprocess the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 3: Split the data (80:20 as mentioned in the paper)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Step 4: Create quantum circuit for QNN
num_features = X_train.shape[1]

# Feature map using angle encoding with RY gates
feature_map = QuantumCircuit(num_features)
params = ParameterVector('x', length=num_features)
for i in range(num_features):
    feature_map.ry(params[i], i)

# Variational circuit
var_circuit = QuantumCircuit(num_features)
var_params = ParameterVector('θ', length=num_features * 3)  # 3 params per qubit

param_index = 0
for i in range(num_features):
    var_circuit.ry(var_params[param_index], i)
    param_index += 1
    var_circuit.rx(var_params[param_index], i)
    param_index += 1
    var_circuit.rz(var_params[param_index], i)
    param_index += 1

# Entanglement (circular)
for i in range(num_features - 1):
    var_circuit.cx(i, i+1)
var_circuit.cx(num_features - 1, 0)

# Step 5: Create QNN using VQC
vqc = VQC(
    feature_map=feature_map,
    ansatz=var_circuit,
    optimizer=COBYLA(maxiter=100),  #  Correct optimizer object
    quantum_instance=Aer.get_backend('statevector_simulator')
)

# Step 6: Train the classifier
vqc.fit(X_train, y_train)

# Step 7: Evaluate the classifier
train_score = vqc.score(X_train, y_train)
test_score = vqc.score(X_test, y_test)

# Print the accuracy
print(f"\n Training Accuracy: {train_score:.2f}")
print(f" Test Accuracy: {test_score:.2f}")

#  Success message
print("\n QNN model for Vlds dataset trained and evaluated successfully!")


c:\Users\gobig\miniconda3\envs\qiskit_ml\lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(



 Training Accuracy: 0.75
 Test Accuracy: 0.65

 QNN model for Vlds dataset trained and evaluated successfully!

 Cell execution time: 25.2010 seconds


Custom dataset - QNN 

In [7]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from qiskit import QuantumCircuit, Aer
from qiskit.utils import algorithm_globals
from qiskit_machine_learning.algorithms import VQC
from qiskit.algorithms.optimizers import COBYLA
from qiskit.circuit import ParameterVector

# Set random seed
algorithm_globals.random_seed = 42

# Step 1: Generate the Custom dataset
X, y = make_classification(n_samples=100, n_features=5, n_informative=3, 
                           n_redundant=0, n_clusters_per_class=1, random_state=42)
# Step 2: Preprocess the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 3: Train-test split (80:20)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Step 4: Build the quantum circuit (Feature map + Variational ansatz)
num_qubits = X_train.shape[1]

# Feature Map: Angle encoding using RY
feature_map = QuantumCircuit(num_qubits)
x_params = ParameterVector("x", num_qubits)
for i in range(num_qubits):
    feature_map.ry(x_params[i], i)

# Variational Circuit: 3 rotational gates per qubit + entanglement
ansatz = QuantumCircuit(num_qubits)
theta = ParameterVector("θ", 3 * num_qubits)
idx = 0
for i in range(num_qubits):
    ansatz.ry(theta[idx], i)
    idx += 1
    ansatz.rx(theta[idx], i)
    idx += 1
    ansatz.rz(theta[idx], i)
    idx += 1

# Circular entanglement
for i in range(num_qubits - 1):
    ansatz.cx(i, i + 1)
ansatz.cx(num_qubits - 1, 0)

# Step 5: VQC model
vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=100),
    quantum_instance=Aer.get_backend("statevector_simulator")
)

# Step 6: Train
vqc.fit(X_train, y_train)

# Step 7: Evaluate
train_score = vqc.score(X_train, y_train)
test_score = vqc.score(X_test, y_test)

# Results
print(f"\nTraining Accuracy: {train_score:.2f}")
print(f"Test Accuracy: {test_score:.2f}")
print("\nQNN model for Custom dataset trained and evaluated successfully!")


c:\Users\gobig\miniconda3\envs\qiskit_ml\lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(



Training Accuracy: 0.71
Test Accuracy: 0.60

QNN model for Custom dataset trained and evaluated successfully!

 Cell execution time: 26.0116 seconds


Adhoc dataset - QNN 

In [8]:
import numpy as np
from sklearn.datasets import make_multilabel_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from qiskit import QuantumCircuit, Aer
from qiskit.utils import algorithm_globals
from qiskit_machine_learning.algorithms import VQC
from qiskit.algorithms.optimizers import COBYLA
from qiskit.circuit import ParameterVector

# Set random seed for reproducibility
algorithm_globals.random_seed = 42

# Step 1: Generate Adhoc dataset (different from Vlds via random_state)
X, y = make_multilabel_classification(n_samples=100, n_features=5, n_classes=1, 
                                      n_labels=1, random_state=99)
y = y.ravel()  # Flatten to 1D array

# Step 2: Preprocess the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 3: Split the data (80:20 split)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Step 4: Create quantum circuit for QNN
num_features = X_train.shape[1]

# Feature map using angle encoding with RY gates
feature_map = QuantumCircuit(num_features)
params = ParameterVector('x', length=num_features)
for i in range(num_features):
    feature_map.ry(params[i], i)

# Variational circuit (Ansatz)
var_circuit = QuantumCircuit(num_features)
var_params = ParameterVector('θ', length=num_features * 3)

param_index = 0
for i in range(num_features):
    var_circuit.ry(var_params[param_index], i)
    param_index += 1
    var_circuit.rx(var_params[param_index], i)
    param_index += 1
    var_circuit.rz(var_params[param_index], i)
    param_index += 1

# Add entanglement (circular)
for i in range(num_features - 1):
    var_circuit.cx(i, i + 1)
var_circuit.cx(num_features - 1, 0)

# Step 5: Create and configure QNN using VQC
vqc = VQC(
    feature_map=feature_map,
    ansatz=var_circuit,
    optimizer=COBYLA(maxiter=100),
    quantum_instance=Aer.get_backend('statevector_simulator')
)

# Step 6: Train the QNN classifier
vqc.fit(X_train, y_train)

# Step 7: Evaluate the classifier
train_score = vqc.score(X_train, y_train)
test_score = vqc.score(X_test, y_test)

# Output results
print(f"\n Training Accuracy (Adhoc): {train_score:.2f}")
print(f" Test Accuracy (Adhoc): {test_score:.2f}")
print("\n QNN model for Adhoc dataset trained and evaluated successfully!")


c:\Users\gobig\miniconda3\envs\qiskit_ml\lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(



 Training Accuracy (Adhoc): 0.65
 Test Accuracy (Adhoc): 0.45

 QNN model for Adhoc dataset trained and evaluated successfully!

 Cell execution time: 25.2639 seconds


Diabetes dataset - QNN 

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from qiskit import QuantumCircuit, Aer
from qiskit.utils import algorithm_globals
from qiskit_machine_learning.algorithms import VQC
from qiskit.algorithms.optimizers import COBYLA
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit.circuit import ParameterVector

# Set random seed
algorithm_globals.random_seed = 42

# Step 1: Load 100 records from the diabetes dataset
file_path = r"C:\Users\gobig\neural_network\diabetes_dataset\diabetes.csv"
df = pd.read_csv(file_path)
df = df.head(100)

# Step 2: Select features and target
feature_columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin']
target_column = 'Outcome'

X = df[feature_columns]
y = df[target_column]

# Step 3: Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 4: Train-test split (80:20)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

#  Fix KeyError: Ensure labels are NumPy arrays
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

# Step 5: Create custom feature map (RY encoding)
num_features = X_train.shape[1]
feature_map = QuantumCircuit(num_features)
params = ParameterVector('x', length=num_features)
for i in range(num_features):
    feature_map.ry(params[i], i)

# Step 6: Create variational circuit (RealAmplitudes or custom)
var_circuit = QuantumCircuit(num_features)
var_params = ParameterVector('θ', length=num_features * 3)

param_index = 0
for i in range(num_features):
    var_circuit.ry(var_params[param_index], i)
    param_index += 1
    var_circuit.rx(var_params[param_index], i)
    param_index += 1
    var_circuit.rz(var_params[param_index], i)
    param_index += 1

# Entanglement (circular)
for i in range(num_features - 1):
    var_circuit.cx(i, i + 1)
var_circuit.cx(num_features - 1, 0)

# Step 7: Create and train VQC
vqc = VQC(
    feature_map=feature_map,
    ansatz=var_circuit,
    optimizer=COBYLA(maxiter=100),
    quantum_instance=Aer.get_backend('statevector_simulator')
)

vqc.fit(X_train, y_train)

# Step 8: Evaluate model
train_accuracy = vqc.score(X_train, y_train)
test_accuracy = vqc.score(X_test, y_test)

# Step 9: Print result
print(f"\nTraining Accuracy: {train_accuracy:.2f}")
print(f"Test Accuracy: {test_accuracy:.2f}")
print("\nQNN model for Diabetes dataset trained and evaluated successfully!")


c:\Users\gobig\miniconda3\envs\qiskit_ml\lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(



Training Accuracy: 0.66
Test Accuracy: 0.75

QNN model for Diabetes dataset trained and evaluated successfully!

 Cell execution time: 25.3764 seconds


Thyroid dataset - QNN 50 records 

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from qiskit import Aer, QuantumCircuit
from qiskit.utils import algorithm_globals
from qiskit_machine_learning.algorithms import VQC
from qiskit.algorithms.optimizers import COBYLA
from qiskit.circuit import ParameterVector

# Set random seed
algorithm_globals.random_seed = 42

# Load the dataset
file_path = r"C:\Users\gobig\neural_network\thyroid_dataset\thyroidDF.csv"
df = pd.read_csv(file_path)

# Convert 'on_thyroxine' column to numeric: 't' -> 1, 'f' -> 0
df['on_thyroxine'] = df['on_thyroxine'].map({'t': 1, 'f': 0})

# Define features and target
features = ['TSH', 'T3', 'TT4', 'FTI', 'on_thyroxine']
target_column = 'target'

# Drop rows with missing values in selected columns
df = df.dropna(subset=features + [target_column])

# Limit to first 50 samples (with stratified balance if needed)
df = df.sample(n=50, random_state=42)

X = df[features]
y = df[target_column].values

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Feature map
num_qubits = X.shape[1]
x = ParameterVector('x', num_qubits)
feature_map = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    feature_map.ry(x[i], i)

# Variational ansatz
theta = ParameterVector('θ', num_qubits * 3)
ansatz = QuantumCircuit(num_qubits)
idx = 0
for i in range(num_qubits):
    ansatz.ry(theta[idx], i)
    idx += 1
    ansatz.rx(theta[idx], i)
    idx += 1
    ansatz.rz(theta[idx], i)
    idx += 1
for i in range(num_qubits - 1):
    ansatz.cx(i, i + 1)
ansatz.cx(num_qubits - 1, 0)

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []

for fold, (train_idx, test_idx) in enumerate(cv.split(X_scaled, y), 1):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    vqc = VQC(
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=COBYLA(maxiter=100),
        quantum_instance=Aer.get_backend('statevector_simulator')
    )

    vqc.fit(X_train, y_train)
    acc = vqc.score(X_test, y_test)
    accuracies.append(acc)
    print(f"Fold {fold} Accuracy: {acc:.2f}")

print(f"\nAverage CV Accuracy over 5 folds: {np.mean(accuracies):.2f}")
print("QNN model for Thyroid dataset trained and evaluated successfully!")

c:\Users\gobig\miniconda3\envs\qiskit_ml\lib\site-packages\sklearn\model_selection\_split.py:700: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\gobig\miniconda3\envs\qiskit_ml\lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Fold 1 Accuracy: 0.60


c:\Users\gobig\miniconda3\envs\qiskit_ml\lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Fold 2 Accuracy: 0.40


c:\Users\gobig\miniconda3\envs\qiskit_ml\lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Fold 3 Accuracy: 0.60


c:\Users\gobig\miniconda3\envs\qiskit_ml\lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Fold 4 Accuracy: 0.60


c:\Users\gobig\miniconda3\envs\qiskit_ml\lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


Fold 5 Accuracy: 0.50

Average CV Accuracy over 5 folds: 0.54
QNN model for Thyroid dataset trained and evaluated successfully!

 Cell execution time: 61.5422 seconds


Thyroid dataset - QNN 100 records 

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from qiskit import Aer, QuantumCircuit
from qiskit.utils import algorithm_globals
from qiskit_machine_learning.algorithms import VQC
from qiskit.algorithms.optimizers import COBYLA
from qiskit.circuit import ParameterVector

# Set random seed
algorithm_globals.random_seed = 42

# Load the dataset
file_path = r"C:\Users\gobig\neural_network\thyroid_dataset\thyroidDF.csv"
df = pd.read_csv(file_path)

# Convert 'on_thyroxine' column to numeric: 't' -> 1, 'f' -> 0
df['on_thyroxine'] = df['on_thyroxine'].map({'t': 1, 'f': 0})

# Define features and target
features = ['TSH', 'T3', 'TT4', 'FTI', 'on_thyroxine']
target_column = 'target'  

# Drop rows with missing values in selected columns
df = df.dropna(subset=features + [target_column])

X = df[features]
y = df[target_column]

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

# Feature map
num_qubits = X.shape[1]
x = ParameterVector('x', num_qubits)
feature_map = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    feature_map.ry(x[i], i)

# Variational ansatz
theta = ParameterVector('θ', num_qubits * 3)
ansatz = QuantumCircuit(num_qubits)
idx = 0
for i in range(num_qubits):
    ansatz.ry(theta[idx], i)
    idx += 1
    ansatz.rx(theta[idx], i)
    idx += 1
    ansatz.rz(theta[idx], i)
    idx += 1
for i in range(num_qubits - 1):
    ansatz.cx(i, i + 1)
ansatz.cx(num_qubits - 1, 0)

# VQC model
vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=100),
    quantum_instance=Aer.get_backend('statevector_simulator')
)

# Train
vqc.fit(X_train, y_train)

# Evaluate
train_acc = vqc.score(X_train, y_train)
test_acc = vqc.score(X_test, y_test)

print(f"QNN for Thyroid Dataset")
print(f"Training Accuracy: {train_acc:.2f}")
print(f"Test Accuracy: {test_acc:.2f}")


c:\Users\gobig\miniconda3\envs\qiskit_ml\lib\site-packages\sklearn\preprocessing\_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


QNN for Thyroid Dataset
Training Accuracy: 0.62
Test Accuracy: 0.62

 Cell execution time: 6965.5340 seconds
